# AIC — Notebook 04: ASR Extraction (Faster-Whisper)

Transcribes audio from all raw videos and aligns to keyframe timestamps.

**GPU required:** T4 (16GB) for large-v3 in float16  
**Estimated time:** ~3-6 hours for full dataset

**Output:** `/kaggle/working/subtitles/L{XX}_{V}.json`

In [ ]:
import subprocess, sys, os
GITHUB_REPO = "https://github.com/YOUR_USERNAME/AIC_System.git"
REPO_DIR = "/kaggle/working/AIC_System"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git","clone","--depth","1",GITHUB_REPO,REPO_DIR],check=True)
else:
    subprocess.run(["git","-C",REPO_DIR,"pull"],check=True)
sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable,"-m","pip","install","-q","-r",f"{REPO_DIR}/requirements.txt"],check=True)
print('Setup complete.')

In [ ]:
from pathlib import Path
DATASET_SLUG = "your-username/aic-hcmc-data"  # ← change
DATASET_NAME = DATASET_SLUG.split('/')[-1]
DATASET_PATH = Path(f"/kaggle/input/{DATASET_NAME}")
MAP_KF_DIR   = DATASET_PATH / "map-keyframes-aic25-b1" / "map-keyframes"
OUTPUT_DIR   = Path("/kaggle/working/subtitles")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
csv_files = sorted(MAP_KF_DIR.glob("*.csv"))
print(f'{len(csv_files)} videos to transcribe')

In [ ]:
from src.feature_extractors.asr_extractor import ASRExtractor
extractor = ASRExtractor(
    model_size='large-v3',
    device='cuda',
    compute_type='float16',
    language='vi',
)
extractor.load()
print('Faster-Whisper large-v3 ready.')

In [ ]:
from tqdm import tqdm
errors = []
for csv_path in tqdm(csv_files, desc='ASR Extraction'):
    video_id = csv_path.stem
    # Find corresponding video file
    batch_id = video_id.split('_')[0]
    video_path = DATASET_PATH / f"Videos_{batch_id}_a" / "video" / f"{video_id}.mp4"
    if not video_path.exists():
        errors.append((video_id, 'video not found'))
        continue
    try:
        extractor.extract_video(
            video_id=video_id,
            video_path=str(video_path),
            map_keyframes_csv=str(csv_path),
            output_dir=str(OUTPUT_DIR),
            overwrite=False,
        )
    except Exception as e:
        errors.append((video_id, str(e)))
        print(f'ERROR: {video_id}: {e}')
print(f'Done. Errors: {len(errors)}')
print(f'Output files: {len(list(OUTPUT_DIR.glob("*.json")))}')